# Setup inicial — Servicio de viento (ArcGIS Enterprise)

Este notebook se ejecuta **una sola vez** para:

1. Consultar los 3 Feature Services de viento (Ecuador, Panamá, Colombia).
2. Interpolar velocidad/dirección a una grilla U/V.
3. Generar el primer ráster, correctamente etiquetado como `Vector-UV`
   (corrige el uso de `CompositeBands_management` de la guía original,
   que apilaba bandas sin etiquetar el tipo de origen).
4. Crear el Mosaic Dataset en la carpeta registrada con tu ArcGIS Server.
5. Agregar el primer ráster al mosaic dataset.

El paso de publicar el Image Service la primera vez sigue siendo manual
(ArcGIS Pro → Compartir como capa web), explicado al final de este
notebook. Para la actualización recurrente usa
`02_actualizacion_periodica_viento.ipynb`.

**Requisitos del entorno** (ver README, sección 11):
- `arcpy` con extensión **Image Analyst** o **Spatial Analyst** (para `VectorField`)
- Paquetes `requests`, `scipy`, `numpy` disponibles en el entorno activo
- Acceso de escritura a la carpeta registrada con tu ArcGIS Server
- Entorno con `arcpy`: ArcGIS Pro, o ArcGIS Notebook Server federado con tu Enterprise

In [ ]:
import arcpy
import numpy as np
import requests
from datetime import datetime
from scipy.interpolate import griddata

# VectorField vive en arcpy.ia (Image Analyst) o arcpy.sa (Spatial Analyst).
# Se intenta primero Image Analyst y se cae a Spatial Analyst si no esta disponible.
try:
    from arcpy.ia import VectorField
    print("Usando arcpy.ia.VectorField (extension Image Analyst)")
except ImportError:
    from arcpy.sa import VectorField
    print("Usando arcpy.sa.VectorField (extension Spatial Analyst)")

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURACION - ajusta estos valores a tu entorno antes de ejecutar.
# Mantenlos iguales en 02_actualizacion_periodica_viento.ipynb.
# ---------------------------------------------------------------------------

SERVICIOS = {
    "Ecuador":  "<FEATURE_SERVICE_URL_ECUADOR>",
    "Panama":   "<FEATURE_SERVICE_URL_PANAMA>",
    "Colombia": "<FEATURE_SERVICE_URL_COLOMBIA>",
}

CAMPO_LAT = "<CAMPO_LATITUD>"           # ej. "Latitud"
CAMPO_LON = "<CAMPO_LONGITUD>"          # ej. "Longitud"
CAMPO_VEL = "<CAMPO_VELOCIDAD_VIENTO>"  # ej. "Velocidad_del_viento__m_s_"
CAMPO_DIR = "<CAMPO_DIRECCION_VIENTO>"  # ej. "Direccion_del_viento____"

# Carpeta registrada como data store con tu ArcGIS Server (ruta UNC, no local)
CARPETA_REGISTRADA = r"<RUTA_CARPETA_REGISTRADA>"  # ej. r"\\servidor\data_registrada\viento"

GDB_NOMBRE = "VientoGDB"          # sin extension .gdb
MOSAICO_NOMBRE = "Viento_UV_Mosaico"
SPATIAL_REFERENCE = arcpy.SpatialReference(4326)

GDB_DESTINO = rf"{CARPETA_REGISTRADA}\{GDB_NOMBRE}.gdb"
MOSAICO_RUTA = rf"{GDB_DESTINO}\{MOSAICO_NOMBRE}"

## Paso 1 — Consultar los Feature Services (velocidad y dirección)

In [ ]:
def consultar_puntos(url_servicio):
    params = {
        "where": "1=1",
        "outFields": f"{CAMPO_LAT},{CAMPO_LON},{CAMPO_VEL},{CAMPO_DIR}",
        "returnGeometry": "false",
        "f": "json",
        "resultRecordCount": 2000,
    }
    r = requests.get(f"{url_servicio}/query", params=params, timeout=30)
    r.raise_for_status()
    out = []
    for feat in r.json().get("features", []):
        a = feat["attributes"]
        if None not in (a.get(CAMPO_LAT), a.get(CAMPO_LON), a.get(CAMPO_VEL), a.get(CAMPO_DIR)):
            out.append((a[CAMPO_LAT], a[CAMPO_LON], a[CAMPO_VEL], a[CAMPO_DIR]))
    return out


todos = []
for pais, url in SERVICIOS.items():
    puntos_pais = consultar_puntos(url)
    print(f"{pais}: {len(puntos_pais)} puntos")
    todos.extend(puntos_pais)

print(f"Total de puntos combinados: {len(todos)}")

lat_arr = np.array([p[0] for p in todos])
lon_arr = np.array([p[1] for p in todos])
vel_arr = np.array([p[2] for p in todos])
dir_arr = np.array([p[3] for p in todos])

## Paso 2 — Interpolar a grilla U/V

In [ ]:
dir_rad = np.deg2rad(dir_arr)
U = -vel_arr * np.sin(dir_rad)
V = -vel_arr * np.cos(dir_rad)

lon_min, lon_max = lon_arr.min() - 0.5, lon_arr.max() + 0.5
lat_min, lat_max = lat_arr.min() - 0.5, lat_arr.max() + 0.5
grid_lon, grid_lat = np.mgrid[lon_min:lon_max:300j, lat_min:lat_max:300j]

grid_U = griddata((lon_arr, lat_arr), U, (grid_lon, grid_lat), method="cubic")
grid_V = griddata((lon_arr, lat_arr), V, (grid_lon, grid_lat), method="cubic")

for g, fuente in [(grid_U, U), (grid_V, V)]:
    mask = np.isnan(g)
    if mask.any():
        g[mask] = griddata((lon_arr, lat_arr), fuente, (grid_lon[mask], grid_lat[mask]), method="nearest")

banda_U = np.flipud(grid_U.T).astype("float32")
banda_V = np.flipud(grid_V.T).astype("float32")

print(f"Grilla generada: {banda_U.shape}")

## Paso 3 — Generar el ráster y etiquetarlo como `Vector-UV` (corregido)

La guía original usaba `arcpy.CompositeBands_management`, que solo apila
las dos bandas sin marcar el tipo de origen del raster. Aquí se usa la
función raster `VectorField`, que compone U y V **y además etiqueta** el
resultado como `Vector-UV` — el requisito que `FlowRenderer` exige en el
cliente JS (ver README, sección 6.1). Se guarda en formato **CRF**, que
Esri documenta como el más apropiado para preservar el tipo Vector-UV y
para mejor rendimiento de lectura.

In [ ]:
punto_origen = arcpy.Point(lon_min, lat_min)
cell_size_x = (lon_max - lon_min) / banda_U.shape[1]
cell_size_y = (lat_max - lat_min) / banda_U.shape[0]

raster_U = arcpy.NumPyArrayToRaster(banda_U, punto_origen, cell_size_x, cell_size_y)
raster_V = arcpy.NumPyArrayToRaster(banda_V, punto_origen, cell_size_x, cell_size_y)

arcpy.management.DefineProjection(raster_U, SPATIAL_REFERENCE)
arcpy.management.DefineProjection(raster_V, SPATIAL_REFERENCE)

# input_data_type="Vector-UV" porque nuestras bandas ya son U y V (no magnitud/direccion)
vector_field_raster = VectorField(raster_U, raster_V, "Vector-UV", "Geographic", "Vector-UV")

fecha_str = datetime.now().strftime("%Y%m")
salida_inicial = rf"{CARPETA_REGISTRADA}\viento_UV_{fecha_str}.crf"

vector_field_raster.save(salida_inicial)
arcpy.management.DefineProjection(salida_inicial, SPATIAL_REFERENCE)

print(f"Raster Vector-UV generado en: {salida_inicial}")

In [ ]:
# Verificacion basica. La confirmacion visual del tipo Vector-UV se hace en
# ArcGIS Pro: panel Raster Properties > pestana "Vector Field".
r_verif = arcpy.Raster(salida_inicial)
print(f"Bandas: {r_verif.bandCount}")  # debe ser 2
print(f"Extension: {r_verif.extent}")

## Paso 4 — Crear el Mosaic Dataset (solo la primera vez)

Las siguientes celdas comprueban `arcpy.Exists` antes de crear, así que
son seguras de re-ejecutar si el notebook se interrumpe. Aun así, este
notebook en general **no debe programarse**; es para la puesta en marcha
inicial únicamente.

In [ ]:
if not arcpy.Exists(GDB_DESTINO):
    arcpy.management.CreateFileGDB(CARPETA_REGISTRADA, GDB_NOMBRE)
    print(f"Geodatabase creada: {GDB_DESTINO}")
else:
    print(f"Geodatabase ya existe: {GDB_DESTINO}")

if not arcpy.Exists(MOSAICO_RUTA):
    arcpy.management.CreateMosaicDataset(
        in_workspace=GDB_DESTINO,
        in_mosaicdataset_name=MOSAICO_NOMBRE,
        coordinate_system=SPATIAL_REFERENCE,
        num_bands=2,
        pixel_type="32_BIT_FLOAT",
    )
    print(f"Mosaic dataset creado: {MOSAICO_RUTA}")
else:
    print(f"Mosaic dataset ya existe: {MOSAICO_RUTA}")

In [ ]:
arcpy.management.AddRastersToMosaicDataset(
    in_mosaic_dataset=MOSAICO_RUTA,
    raster_type="Raster Dataset",
    input_path=salida_inicial,
)
print("Primer raster agregado al mosaic dataset.")

## Paso 5 — Publicar como Image Service (acción manual, una sola vez)

Este paso se hace desde la interfaz de ArcGIS Pro, no por código, porque
es una acción única de puesta en marcha:

1. Agrega `Viento_UV_Mosaico` a un mapa en ArcGIS Pro.
2. Clic derecho sobre la capa → **Compartir como → Capa web** → tipo **Imagery**.
3. Elige **"Referenciar datos registrados"** (no "Copiar todos los datos") —
   esto es lo que permite actualizar sin republicar más adelante.
4. Selecciona tu servidor Enterprise como destino (no ArcGIS Online), y en
   la carpeta de destino del servicio elige **`Viento-SoX`** — la carpeta
   que ya creaste en ArcGIS Server Manager (Manage Services → Folders) —
   en vez de dejarlo en la raíz del sitio.
5. Publica y copia la URL resultante (termina en `/ImageServer`, con la
   forma `.../rest/services/Viento-SoX/<nombre_servicio>/ImageServer`).
6. **Nuevo respecto a la guía original:** en ArcGIS Server Manager, abre
   las propiedades del servicio y activa **`hasLiveData: true`** — es un
   prerrequisito documentado para que `RefreshService` funcione en el
   notebook de actualización periódica (ver README, sección 6.3).

Guarda la URL del Image Service resultante: la necesitas en
`02_actualizacion_periodica_viento.ipynb` y en la app web (README, sección 8).

**Siguiente paso:** usa `02_actualizacion_periodica_viento.ipynb` para la
actualización recurrente, y prográmalo en ArcGIS Notebook Server (o Task
Scheduler si aún no lo tienes instalado) según la sección 7.1 del README.